## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/chennai/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/chennai/


In [6]:
##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Chennai'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Chennai District, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [7]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(784865, 2)


,customer_id,usecase_tag
0,650f03a04a55e4432dd8dc04,residential
1,630cb98aaad3f50b1ace906f,food


### Active customer (RR-customers)

In [8]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/chennai_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(690685, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,65eb549f336a6f47916f9bb9,MALE,30.0,NaN,26.0,6.0,4.0,PHH,7,HOOK,MEDIUM_INCOME,ONLY_LINK,MEDIUM_DISTANCE,PS,4.0,d. high rpc
1,648ea58a19c42eda45912e0a,FEMALE,294.0,NaN,24.0,10.0,5.0,PHH,26,COMMITTED,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN,UNKNOWN,5.0,d. high rpc


In [9]:
df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
                                        how='left', on=['customer_id'])

### customer installed apps

In [10]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(689917, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,5737c6e0ddbec2203f733329,"['telegram', 'ludo king', 'instagram', 'zoom',...",32,"['Tools', 'Social', 'News', 'Delivery_Food', '...",12,"['Gaming', 'Ride haling', 'Office', 'Rest']",4,0,0,1,0,1,1,1
1,573f28f39b0ffc283676f277,"['instagram', 'zoom', 'zomato', 'messenger', '...",36,"['Tools', 'Social', 'News', 'Health', 'Deliver...",15,"['Office', 'Ride haling', 'Finance_Investment'...",4,0,0,0,1,1,1,1


### Customer app & cat explode mapping

In [11]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,65eb549f336a6f47916f9bb9,Rest


In [12]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

689917

In [13]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['News' 'Social' 'Finance_Transactions' 'OTT' 'Tools' 'Ecommerce'
 'Travel_Ridehailing' 'Delivery_Food' 'Streaming_Music' 'Delivery_Grocery'
 'Office' 'Entertainment' 'Wearable' 'Gaming' 'Travel_Bookings'
 'Finance_Investment' 'Driver_App' 'Educational' 'Finance_News'
 'Fantasy_Sports' 'Dating' 'Health' 'Devotional']


In [14]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Ride haling' 'Office' 'Gaming' 'Finance_Investment' 'Driver_App'
 'Educational']


In [15]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [17]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,407348,59.04
1,Ride haling,uber,304166,44.09
2,Ride haling,namma yatri,48650,7.05
3,Ride haling,in drive,19225,2.79
4,Ride haling,quick ride,7285,1.06
5,Ride haling,driveu driver app,1336,0.19
6,Ride haling,blusmart,1214,0.18
7,Ride haling,jugnoo,240,0.03
8,Ride haling,quickride,221,0.03
9,Office,zoom,187852,27.23


### Merge raw data

In [18]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(689917, 30)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,65eb549f336a6f47916f9bb9,MALE,30.0,NaN,26.0,6.0,4.0,PHH,7,HOOK,MEDIUM_INCOME,ONLY_LINK,MEDIUM_DISTANCE,PS,4.0,d. high rpc,food,"['instagram', 'facebook', 'google news', 'mess...",6,"['Finance_Transactions', 'Social', 'News']",3,['Rest'],1,0,0,0,0,0,0,1


#### Derived columns

In [19]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    616776.000000
mean          3.018342
std           3.565214
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max         114.000000
Name: rpc, dtype: float64

In [20]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [21]:
## app_count_bucket

df_customer_data.app_count.describe()

count    689917.000000
mean         17.520132
std           8.774400
min           1.000000
25%          11.000000
50%          16.000000
75%          23.000000
max          85.000000
Name: app_count, dtype: float64

In [22]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [23]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    689917.000000
mean          2.969531
std           1.202553
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

In [24]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 4 , '2-3','Above 3'))

In [25]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    689805.000000
mean        647.275610
std         533.557972
min           2.000000
25%         216.000000
50%         556.000000
75%         869.000000
max        2874.000000
Name: rapido_age, dtype: float64

#### Merge

In [26]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '65eb549f336a6f47916f9bb9'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y
0,65eb549f336a6f47916f9bb9,MALE,30.0,NaN,26.0,6.0,4.0,PHH,7,HOOK,MEDIUM_INCOME,ONLY_LINK,MEDIUM_DISTANCE,PS,4.0,4.0,food,"['instagram', 'facebook', 'google news', 'mess...",6,"['Finance_Transactions', 'Social', 'News']",3,['Rest'],1,0,0,0,0,0,0,1,d. HIGH,1-5,1,1,Rest


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [27]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,689299,99.91
1,Ride haling,486557,70.52
2,Office,397938,57.68
3,Finance_Investment,196311,28.45
4,Educational,142108,20.60
5,Gaming,70471,10.21
6,Driver_App,66046,9.57


#### LTR-Segment

In [28]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,126539,18.34
1,LTR 0,13425,1.95
2,PHH,549721,79.68
3,UNKNOWN,232,0.03


In [29]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                \
categories_l2 Driver_App Educational Finance_Investment Gaming  Office   
ltr_segment                                                              
HH                 11246       23196              27841  12147   62827   
LTR 0               1592        2256               2472   1443    5771   
PHH                53184      116605             165924  56861  329192   

                                   
categories_l2    Rest Ride haling  
ltr_segment                        
HH             126394       72189  
LTR 0           13399        7083  
PHH            549275      407137

In [30]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [31]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                   8.89       18.33              22.00   9.60  49.65  99.89   
LTR 0               11.86       16.80              18.41  10.75  42.99  99.81   
PHH                  9.67       21.21              30.18  10.34  59.88  99.92   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  57.05  
LTR 0               52.76  
PHH                 74.06

#### Other testing

In [32]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [33]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,72189,14.84
1,LTR 0,7083,1.46
2,PHH,407137,83.68
3,UNKNOWN,148,0.03


In [34]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                        \
app_name     blusmart driveu driver app in drive jugnoo namma yatri     ola   
ltr_segment                                                                   
HH                110               121     1755     25        4994   59476   
LTR 0              11                10      308      1         438    5581   
PHH              1093              1205    17159    214       43210  342165   

                                          
app_name    quick ride quickride    uber  
ltr_segment                               
HH                 778        56   39672  
LTR 0               58         5    3837  
PHH               6447       160  260562

In [35]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [36]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 0.15              0.17     2.43   0.03        6.92  82.39   
LTR 0              0.16              0.14     4.35   0.01        6.18  78.79   
PHH                0.27              0.30     4.21   0.05       10.61  84.04   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                1.08      0.08  54.96  
LTR 0             0.82      0.07  54.17  
PHH               1.58      0.04  64.00

##### Explode - Office

In [37]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
2,627e3c90c75b90085b09860d,microsoft teams,microsoft teams,Office,Office,PHH
4,64c9c57168211d737da5e1d3,onedrive,onedrive,Office,Office,PHH


In [38]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
2,627e3c90c75b90085b09860d,microsoft teams,microsoft teams,Office,Office,PHH,secondary_office_app
4,64c9c57168211d737da5e1d3,onedrive,onedrive,Office,Office,PHH,secondary_office_app


In [39]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'slack', 'google analytics', 'github',
       'zoho meeting', 'zoho mail', 'trello', 'jira', 'asana', 'cisco',
       'miro', 'confluence'], dtype=object)

In [40]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,62827,15.79
1,LTR 0,5771,1.45
2,PHH,329192,82.72
3,UNKNOWN,148,0.04


In [41]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                     \
app_name        asana  cisco confluence dropbox  github google analytics   
ltr_segment                                                                
HH               52.0   43.0       49.0   782.0   644.0             83.0   
LTR 0             3.0    7.0        NaN    65.0    51.0              7.0   
PHH             489.0  245.0      267.0  5190.0  3440.0            661.0   

                                                                               \
app_name    google authenticator intune  company portal    jira microsoft 365   
ltr_segment                                                                     
HH                        4554.0                 9084.0   209.0       23377.0   
LTR 0                      335.0                  682.0    14.0        2057.0   
PHH                      30810.0                55550.0  1546.0      128683.0   

                                                                       \
app_name    microsoft teams   miro ms authenticator nishtha  onedrive   
ltr_segment                                                             
HH                  21752.0   39.0          12533.0     NaN   27221.0   
LTR 0                1828.0    5.0            977.0     NaN    2579.0   
PHH                127654.0  223.0          74544.0     2.0  137638.0   

                                                                              \
app_name      outlook pocket    slack trello    webex zoho mail zoho meeting   
ltr_segment                                                                    
HH            23366.0   75.0   1343.0  106.0   3877.0     594.0        478.0   
LTR 0          1959.0    8.0    106.0    4.0    330.0      51.0         55.0   
PHH          132732.0  547.0  10299.0  963.0  22646.0    4631.0       2508.0   

                       
app_name         zoom  
ltr_segment            
HH            28225.0  
LTR 0          2576.0  
PHH          156975.0

In [42]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [43]:
print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual app.


customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.08  0.07       0.08    1.24   1.03             0.13   
LTR 0              0.05  0.12        NaN    1.13   0.88             0.12   
PHH                0.15  0.07       0.08    1.58   1.04             0.20   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          7.25                  14.46  0.33         37.21   
LTR 0                       5.80                  11.82  0.24         35.64   
PHH                         9.36                  16.87  0.47         39.09   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    34.62  0.06            19.95     NaN    43.33   37.19   
LTR 0                 31.68  0.09            16.93     NaN    44.69   33.95   
PHH                   38.78  0.07            22.64     0.0    41.81   40.32   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.12  2.14   0.17  6.17      0.95         0.76  44.92  
LTR 0         0.14  1.84   0.07  5.72      0.88         0.95  44.64  
PHH           0.17  3.13   0.29  6.88      1.41         0.76  47.68

In [44]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     11796                62024
LTR 0                    926                 5693
PHH                    73922               324738

In [45]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     18.78                98.72
LTR 0                  16.05                98.65
PHH                    22.46                98.65

##### Explode - Education

In [46]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
33,61a63dc40d37fdceb34b4c0a,udemy,udemy,Educational,Educational,PHH
38,6128bce1890209dab95a0ac9,google classroom,google classroom,Educational,Educational,PHH


In [47]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
33,61a63dc40d37fdceb34b4c0a,udemy,udemy,Educational,Educational,PHH,secondary_education_app
38,6128bce1890209dab95a0ac9,google classroom,google classroom,Educational,Educational,PHH,code_education_app


In [48]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['google classroom', 'byju', 'microsoft math solver', 'aakash',
       'simplilearn', 'moodle', 'diksha', 'physics wallah', 'vedantu',
       'mycbseguide', 'e-pg pathshala', 'chegg study', 'vidyakul'],
      dtype=object)

In [49]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,23196,16.32
1,LTR 0,2256,1.59
2,PHH,116605,82.05
3,UNKNOWN,51,0.04


In [50]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                         \
app_name       aakash  brainly     byju caclubindia chegg study colegeduniya   
ltr_segment                                                                    
HH              136.0   3145.0   3034.0         9.0        50.0         23.0   
LTR 0             5.0    287.0    383.0         NaN         4.0          2.0   
PHH             482.0  15418.0  13334.0       116.0       265.0         82.0   

                                                                         \
app_name    collegedekho course hero coursera  diksha doubtnut duolingo   
ltr_segment                                                               
HH                   3.0         7.0   1253.0   480.0    210.0   4318.0   
LTR 0                NaN         2.0    124.0    46.0     27.0    403.0   
PHH                  5.0        27.0   7688.0  2165.0    839.0  23318.0   

                                                                             \
app_name    e-pg pathshala     edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                   
HH                    74.0   168.0   72.0     2.0           262.0     212.0   
LTR 0                  6.0    14.0    5.0     NaN            20.0      18.0   
PHH                  360.0  1255.0  309.0     5.0          1363.0    1715.0   

                                                                               \
app_name    google classroom happify ignou e-content khan academy meritnation   
ltr_segment                                                                     
HH                    9425.0     1.0            62.0        107.0         NaN   
LTR 0                  889.0     NaN             5.0         13.0         NaN   
PHH                  45313.0    13.0           334.0        647.0         5.0   

                                                                     \
app_name    microsoft math solver  moodle my study life mycbseguide   
ltr_segment                                                           
HH                          131.0   373.0          13.0       149.0   
LTR 0                         9.0    29.0           1.0        14.0   
PHH                         569.0  2054.0          45.0       815.0   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                 94.0     149.0          350.0   75.0           NaN   
LTR 0              10.0       8.0           40.0    8.0           NaN   
PHH               463.0     665.0         1640.0  547.0           1.0   

                                                                           \
app_name    shiksha.com simplilearn stack overflow  swayam toppr    udemy   
ltr_segment                                                                 
HH                 23.0       433.0            5.0  2003.0   3.0   3626.0   
LTR 0               NaN        36.0            NaN   198.0   NaN    299.0   
PHH               123.0      2472.0           27.0  8105.0  16.0  23526.0   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH             1282.0   236.0      2.0        73.0  
LTR 0           156.0    20.0      2.0         8.0  
PHH            6328.0  1077.0     12.0       375.0

In [51]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [52]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                      \
app_name         aakash brainly   byju caclubindia chegg study colegeduniya   
ltr_segment                                                                   
HH                 0.59   13.56  13.08        0.04        0.22         0.10   
LTR 0              0.22   12.72  16.98         NaN        0.18         0.09   
PHH                0.41   13.22  11.44        0.10        0.23         0.07   

                                                                        \
app_name    collegedekho course hero coursera diksha doubtnut duolingo   
ltr_segment                                                              
HH                  0.01        0.03     5.40   2.07     0.91    18.62   
LTR 0                NaN        0.09     5.50   2.04     1.20    17.86   
PHH                 0.00        0.02     6.59   1.86     0.72    20.00   

                                                                           \
app_name    e-pg pathshala   edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                 
HH                    0.32  0.72   0.31    0.01            1.13      0.91   
LTR 0                 0.27  0.62   0.22     NaN            0.89      0.80   
PHH                   0.31  1.08   0.26    0.00            1.17      1.47   

                                                                               \
app_name    google classroom happify ignou e-content khan academy meritnation   
ltr_segment                                                                     
HH                     40.63    0.00            0.27         0.46         NaN   
LTR 0                  39.41     NaN            0.22         0.58         NaN   
PHH                    38.86    0.01            0.29         0.55         0.0   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                           0.56   1.61          0.06        0.64   
LTR 0                        0.40   1.29          0.04        0.62   
PHH                          0.49   1.76          0.04        0.70   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                 0.41      0.64           1.51   0.32           NaN   
LTR 0              0.44      0.35           1.77   0.35           NaN   
PHH                0.40      0.57           1.41   0.47           0.0   

                                                                        \
app_name    shiksha.com simplilearn stack overflow swayam toppr  udemy   
ltr_segment                                                              
HH                 0.10        1.87           0.02   8.64  0.01  15.63   
LTR 0               NaN        1.60            NaN   8.78   NaN  13.25   
PHH                0.11        2.12           0.02   6.95  0.01  20.18   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH               5.53    1.02     0.01        0.31  
LTR 0            6.91    0.89     0.09        0.35  
PHH              5.43    0.92     0.01        0.32

In [53]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        13758                   13570
LTR 0                      1389                    1253
PHH                       65170                   72721

In [54]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        59.31                   58.50
LTR 0                     61.57                   55.54
PHH                       55.89                   62.37

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to move towards PHH.
    - Finance_Investment, Office, Ride haling, Educational, Gaming
- Whenever an app belongs to the app-categories listed below, customers are more likely to be LTR0/HH.
    - Driver App

### Other

#### Service Affinity

In [55]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,12853,1.86
1,LINK_AUTO,62558,9.07
2,LINK_CAB,3615,0.52
3,NO_AFFINITY,42754,6.20
4,ONLY_AUTO,276192,40.03
5,ONLY_CAB,15786,2.29
6,ONLY_LINK,248805,36.06
7,UNKNOWN,27354,3.96


In [56]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers'])

customers                                                \
categories_l2    Driver_App Educational Finance_Investment Gaming  Office   
service_affinity                                                            
AUTO_CAB                788        2714               4067   1216    8443   
LINK_AUTO              7250       12688              18014   6831   34598   
LINK_CAB                459         660               1220    394    2019   
NO_AFFINITY            4940        8290              14641   4488   25695   
ONLY_AUTO             17578       61364              70601  28306  165761   
ONLY_CAB               1283        3046               5073   1564    9565   
ONLY_LINK             30776       48001              75588  24580  137272   

                                      
categories_l2       Rest Ride haling  
service_affinity                      
AUTO_CAB           12836       10190  
LINK_AUTO          62507       44152  
LINK_CAB            3613        2381  
NO_AFFINITY        42726       32577  
ONLY_AUTO         275882      211586  
ONLY_CAB           15768       11416  
ONLY_LINK         248645      157495

In [57]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                6.13       21.12              31.64   9.46  65.69   
LINK_AUTO              11.59       20.28              28.80  10.92  55.31   
LINK_CAB               12.70       18.26              33.75  10.90  55.85   
NO_AFFINITY            11.55       19.39              34.24  10.50  60.10   
ONLY_AUTO               6.36       22.22              25.56  10.25  60.02   
ONLY_CAB                8.13       19.30              32.14   9.91  60.59   
ONLY_LINK              12.37       19.29              30.38   9.88  55.17   

                                     
categories_l2      Rest Ride haling  
service_affinity                     
AUTO_CAB          99.87       79.28  
LINK_AUTO         99.92       70.58  
LINK_CAB          99.94       65.86  
NO_AFFINITY       99.93       76.20  
ONLY_AUTO         99.89       76.61  
ONLY_CAB          99.89       72.32  
ONLY_LINK         99.94       63.30

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [58]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,175306,25.41
1,MEDIUM_DISTANCE,164604,23.86
2,SHORT_DISTANCE,165177,23.94
3,UNKNOWN,184830,26.79


In [59]:
df1 = df_customer_data_explode[~df_customer_data_explode['distance_preference'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,distance_affinity_city[['distance_preference','customers']], how= 'left', on='distance_preference')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2        Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                            
LONG_DISTANCE              9.92       18.97              28.99   9.98  57.17   
MEDIUM_DISTANCE            8.83       20.66              28.10   9.86  57.64   
SHORT_DISTANCE             8.61       22.62              28.08  10.35  58.38   

                                        customers              \
categories_l2         Rest Ride haling Driver_App Educational   
distance_preference                                             
LONG_DISTANCE        99.91       72.35    17395.0     33264.0   
MEDIUM_DISTANCE      99.91       70.05    14538.0     33999.0   
SHORT_DISTANCE       99.90       68.57    14216.0     37363.0   

                                                                     \
categories_l2       Finance_Investment   Gaming    Office      Rest   
distance_preference                                                   
LONG_DISTANCE                  50824.0  17494.0  100225.0  175141.0   
MEDIUM_DISTANCE                46248.0  16230.0   94881.0  164459.0   
SHORT_DISTANCE                 46381.0  17104.0   96428.0  165007.0   

                                 
categories_l2       Ride haling  
distance_preference              
LONG_DISTANCE          126841.0  
MEDIUM_DISTANCE        115303.0  
SHORT_DISTANCE         113268.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [60]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,219418,31.80
1,MALE,446992,64.79
2,OTHERS,2137,0.31
3,UNKNOWN,21370,3.10


In [61]:
df1 = df_customer_data_explode[~df_customer_data_explode['gender'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,gender_city[['gender','customers']], how= 'left', on='gender')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
gender                                                                          
FEMALE               3.88       24.44              22.31  12.19  61.08  99.90   
MALE                12.59       18.59              31.64   9.21  55.98  99.92   
OTHERS               6.36       16.28              10.67  11.23  39.68  99.91   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
gender                                                                         
FEMALE              77.04     8513.0     53632.0            48960.0  26756.0   
MALE                67.24    56288.0     83092.0           141411.0  41185.0   
OTHERS              61.91      136.0       348.0              228.0    240.0   

                                               
categories_l2    Office      Rest Ride haling  
gender                                         
FEMALE         134015.0  219192.0    169048.0  
MALE           250213.0  446624.0    300575.0  
OTHERS            848.0    2135.0      1323.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [62]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,264773,38.38
1,LOW_INCOME,48885,7.09
2,MEDIUM_INCOME,271647,39.37
3,UNKNOWN,104612,15.16


In [63]:
df1 = df_customer_data_explode[~df_customer_data_explode['income_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,income_segment_city[['income_segment','customers']], how= 'left', on='income_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2   Driver_App Educational Finance_Investment Gaming Office   
income_segment                                                            
HIGH_INCOME          10.16       24.66              36.27  12.31  65.97   
LOW_INCOME            7.38       11.41              13.01   5.84  36.09   
MEDIUM_INCOME         9.35       20.50              24.58   9.43  57.37   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
income_segment                                                                
HIGH_INCOME     99.97       78.71    26898.0     65301.0            96028.0   
LOW_INCOME      99.65       52.16     3607.0      5579.0             6359.0   
MEDIUM_INCOME   99.90       69.09    25398.0     55697.0            66779.0   

                                                         
categories_l2    Gaming    Office      Rest Ride haling  
income_segment                                           
HIGH_INCOME     32596.0  174679.0  264687.0    208400.0  
LOW_INCOME       2855.0   17643.0   48716.0     25500.0  
MEDIUM_INCOME   25629.0  155844.0  271371.0    187685.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [64]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,201210,29.16
1,PS,194744,28.23
2,UNKNOWN,293963,42.61


In [65]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                     10.63       20.77              29.46  10.78  57.90   
PS                       9.16       21.02              29.61  10.12  59.81   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.92       72.59    21396.0     41798.0   
PS                 99.93       74.44    17835.0     40934.0   

                                                                               
categories_l2     Finance_Investment   Gaming    Office      Rest Ride haling  
price_sensitivity                                                              
NPS                          59274.0  21692.0  116497.0  201059.0    146050.0  
PS                           57670.0  19717.0  116480.0  194598.0    144975.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [66]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,73052,10.59
1,b. LOW,268491,38.92
2,c. MEDIUM,200286,29.03
3,d. HIGH,147999,21.45


In [67]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO             12.09       20.70              27.67  11.63  54.34  99.89   
b. LOW              10.27       20.03              28.57  10.07  56.47  99.91   
c. MEDIUM            9.23       20.69              28.87  10.08  58.44  99.92   
d. HIGH              7.52       21.46              28.07   9.96  60.49  99.90   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
rpc_bucket                                                                     
a. ZERO             68.61     8835.0     15122.0            20212.0   8493.0   
b. LOW              67.72    27580.0     53772.0            76713.0  27041.0   
c. MEDIUM           70.92    18492.0     41436.0            57815.0  20189.0   
d. HIGH             76.03    11131.0     31763.0            41545.0  14740.0   

                                               
categories_l2    Office      Rest Ride haling  
rpc_bucket                                     
a. ZERO         39693.0   72972.0     50119.0  
b. LOW         151624.0  268256.0    181809.0  
c. MEDIUM      117051.0  200125.0    142047.0  
d. HIGH         89523.0  147857.0    112517.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [68]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,582653,84.45
1,11-15,22453,3.25
2,6-10,74189,10.75
3,Above 15,10622,1.54


In [69]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   88.19       83.77              84.75  84.91  83.61   
11-15                  2.24        3.50               3.21   3.22   3.49   
6-10                   8.40       11.23              10.60  10.46  11.32   
Above 15               1.17        1.50               1.44   1.41   1.58   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               84.45       83.03  
11-15              3.25        3.62  
6-10              10.75       11.62  
Above 15           1.54        1.73

In [70]:
df_customer_data.app_count.describe()

count    689917.000000
mean         17.520132
std           8.774400
min           1.000000
25%          11.000000
50%          16.000000
75%          23.000000
max          85.000000
Name: app_count, dtype: float64

#### Category Count Bucket

In [71]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,84062,12.18
1,2-3,372830,54.04
2,Above 3,233025,33.78


In [72]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.03        0.01               0.03    NaN   0.03   
2-3                        43.77       22.96              25.69  31.21  45.98   
Above 3                    56.20       77.03              74.28  68.79  53.99   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      12.13        0.05  
2-3                    54.06       53.97  
Above 3                33.81       45.99

In [73]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2
0,"{['Ride haling', 'Office', 'Rest']}"
1,"{['Ride haling', 'Office', 'Rest']}"
2,"{['Ride haling', 'Office', 'Rest']}"
3,"{['Ride haling', 'Office', 'Rest']}"
4,"{['Ride haling', 'Finance_Investment', 'Rest']}"
...,...
372825,"{['Office', 'Rest']}"
372826,"{['Ride haling', 'Office', 'Rest']}"
372827,"{['Ride haling', 'Office', 'Rest']}"
372828,"{['Educational', 'Rest']}"


In [74]:
df_customer_data.category_count.describe()

count    689917.000000
mean          2.969531
std           1.202553
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

#### Customer Use-Case 

In [75]:
usecase_tag_city = df_customer_data\
                        .groupby(['usecase_tag'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
usecase_tag_city['% city threshold']=(usecase_tag_city['customers']*100.00/usecase_tag_city.customers.sum()).round(2)
usecase_tag_city.sort_values(by=['usecase_tag'],ascending=[True])

,usecase_tag,customers,% city threshold
0,Unknown,157851,22.88
1,educational,45383,6.58
2,food,28906,4.19
3,govt_amenity,11757,1.70
4,health_and_personal,51202,7.42
5,household_needs,27341,3.96
6,leisure,66618,9.66
7,office,46455,6.73
8,place_of_worship,9749,1.41
9,residential,139489,20.22


In [76]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'usecase_tag'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'usecase_tag'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='usecase_tag' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2       Driver_App Educational Finance_Investment Gaming Office   
usecase_tag                                                                   
Unknown                  26.20       23.41              22.77  22.27  22.31   
educational               5.81        7.52               5.64   6.48   6.50   
food                      4.09        4.13               4.14   4.43   4.17   
govt_amenity              1.90        1.59               1.51   1.62   1.59   
health_and_personal       7.02        7.16               6.96   7.34   7.29   
household_needs           3.59        3.61               3.77   4.15   3.79   
leisure                   8.77        9.73               9.70  10.33   9.79   
office                    6.07        7.05               8.04   6.78   7.53   
place_of_worship          1.24        1.38               1.26   1.20   1.40   
residential              20.65       19.41              19.76  19.68  20.17   
transit_station          14.67       15.00              16.45  15.71  15.46   

                                        
categories_l2         Rest Ride haling  
usecase_tag                             
Unknown              22.88       22.25  
educational           6.58        6.75  
food                  4.19        4.24  
govt_amenity          1.70        1.66  
health_and_personal   7.42        7.61  
household_needs       3.96        4.07  
leisure               9.66        9.86  
office                6.73        6.98  
place_of_worship      1.41        1.46  
residential          20.22       20.63  
transit_station      15.25       14.51

#### AO-NET Funnel

In [77]:
df_customer_data['city'] = 'Chennai, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Chennai, Android"
% fe2rr,46.98
% g2n,71.3
% fe2net,33.49


In [78]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Ride haling,Office,Finance_Investment,Educational,Gaming,Driver_App
% fe2rr,46.97,48.52,47.76,46.65,47.38,46.28,45.8
% g2n,71.3,71.78,72.35,71.73,71.14,68.56,67.86
% fe2net,33.49,34.83,34.55,33.47,33.71,31.73,31.08
